# Lecture 13 — CMake Basics

Today’s goal:

> Understand the minimum CMake mental model needed to work in Autoware metric packages.

You do not need to master CMake now. You only need enough to not get lost.

## 1. Why CMake exists

C++ projects often have many files:

```text
main.cpp
speed_metric.cpp
speed_metric.hpp
ttc_metric.cpp
ttc_metric.hpp
```

The compiler needs to know:

```text
Which .cpp files should be compiled?
Which libraries should be linked?
Where are header files?
Which tests should be built?
```

CMake helps describe this build structure.

## 2. Very tiny CMake example

For a normal small C++ project:

```cmake
cmake_minimum_required(VERSION 3.14)
project(my_metric_project)

add_executable(main
  main.cpp
  speed_metric.cpp
)
```

Meaning:

```text
Build an executable called main
using main.cpp and speed_metric.cpp
```

## 3. Header files are usually not compiled directly

You normally compile `.cpp` files.

```text
Compiled:
  main.cpp
  speed_metric.cpp

Included but not directly compiled:
  speed_metric.hpp
```

The `.hpp` is pulled in by `#include`.

## 4. Autoware / ROS2 uses ament and colcon

Autoware packages are not built with plain `g++` manually.

They usually use:

```text
CMakeLists.txt
package.xml
colcon build
ament_cmake
autoware_cmake
```

For package-level builds, you often run:

```bash
colcon build --packages-select package_name
```

## 5. CMakeLists.txt in metric package style

You may see a pattern like:

```cmake
ament_auto_add_library(${PROJECT_NAME} SHARED
  src/buffer.cpp
  src/base_evaluator.cpp
  src/open_loop_evaluator.cpp
  src/metrics/epdms/subscores/ego_progress.cpp
  src/metrics/epdms/subscores/ttc_within_bound.cpp
)
```

Meaning:

```text
These .cpp files are part of the package library.
```

If you add a new metric `.cpp` file, you usually need to add it to this list.

## 6. Test registration

Tests are also registered in CMake.

Example:

```cmake
ament_add_gtest(
  test_metrics
  test/metrics/test_ego_progress.cpp
  test/metrics/test_ttc_within_bound.cpp
)
```

Meaning:

```text
Build a test executable called test_metrics
using these test source files.
```

If you add a new test file, you usually need to register it.

## 7. Common beginner problem

You create:

```text
src/metrics/epdms/subscores/my_new_metric.cpp
```

But forget to add it to `CMakeLists.txt`.

Then:

```text
File exists, but build ignores it.
```

This is a very classic C++ project trap.

## 8. Minimum workflow when adding a metric

```text
1. Add my_metric.hpp
2. Add my_metric.cpp
3. Add test_my_metric.cpp
4. Register my_metric.cpp in ament_auto_add_library
5. Register test_my_metric.cpp in ament_add_gtest
6. colcon build --packages-select package_name
7. colcon test --packages-select package_name
```

## 9. Build command mental model

Typical package build:

```bash
colcon build   --symlink-install   --packages-select autoware_planning_data_analyzer
```

Typical test command:

```bash
colcon test   --packages-select autoware_planning_data_analyzer   --event-handlers console_direct+
```

Show test results:

```bash
colcon test-result --verbose
```

## 10. Mini exercise: where should the file go?

Suppose you add:

```text
src/metrics/epdms/subscores/speed_limit_compliance.cpp
```

Question 1: where should you register it?

Answer:

```text
ament_auto_add_library(...)
```

Suppose you add:

```text
test/metrics/test_speed_limit_compliance.cpp
```

Question 2: where should you register it?

Answer:

```text
ament_add_gtest(...)
```

## 11. CMake is not the metric logic

Do not confuse:

```text
C++ metric code = actual behavior
CMakeLists.txt = build instructions
package.xml = dependency/package metadata
```

When a build fails, ask:

```text
Is the C++ code wrong?
Is the file not registered?
Is a dependency missing?
Is include path missing?
```

## 12. Part A–C complete

You now covered the beginner foundation through file/build organization:

```text
01. What is a C++ program?
02. Variables and types
03. if / else / comparisons
04. Functions
05. struct
06. std::vector
07. Loops
08. const references
09. Pointers and nullptr
10. std::optional
11. .hpp and .cpp
12. #include and namespaces
13. CMake basics
```

Next stage would be a mini metric project before touching real ROS2/Autoware messages.